# Κεφάλαιο 10: Η προσοχή είναι το μόνο που χρειάζεσαι

> «Η προσοχή είναι το μόνο που χρειάζεσαι.» — **Vaswani κ.ά.**, Google Research, 2017

---

## Τι θα μάθεις

- Γιατί δεν αρκούν οι στατικές διανυσματικές αναπαραστάσεις (embeddings): χρησιμοποιούμε την αγγλική λέξη «bank» επειδή σημαίνει «όχθη» στο «river bank» αλλά «τράπεζα» στο «savings bank»
- Ποια είναι η βασική ιδέα της προσοχής: κάθε token (μονάδα κειμένου) «κοιτάζει» τα άλλα token
- Πώς να φτιάξεις βήμα βήμα έναν μηχανισμό αυτοπροσοχής με ερωτήματα, κλειδιά και τιμές
- Γιατί αλλάζουμε την κλίμακα των βαθμών προσοχής και κρύβουμε τα επόμενα token με μάσκα
- Πώς να περάσεις αποδοτικά από την προσοχή με μία κεφαλή στην προσοχή με πολλαπλές κεφαλές
- Πώς να συνδυάσεις την προσοχή με δίκτυα πρόσθιας τροφοδότησης για να φτιάξεις πλήρη μπλοκ Transformer
- Πώς να απεικονίσεις τα μοτίβα προσοχής που προκύπτουν

---

## Προετοιμασία

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch10.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch10.ipynb).

Πρώτα, εγκατέστησε τα απαραίτητα πακέτα:

In [ ]:
# Εγκατέστησε τα απαραίτητα πακέτα
!pip install -q torch transformers matplotlib

In [ ]:
# ===== Εισαγωγή βιβλιοθηκών =====
import torch                     # PyTorch: πράξεις με τανυστές
import torch.nn as nn            # Επίπεδα νευρωνικών δικτύων
import torch.nn.functional as F  # Μαθηματικές συναρτήσεις (softmax, gelu)
import math                      # Για τη sqrt στην κλιμάκωση των βαθμών προσοχής
import matplotlib.pyplot as plt  # Απεικόνιση
import numpy as np               # Πράξεις με πίνακες

# Βασικά εργαλεία που θα χρησιμοποιήσουμε:
# - nn.Linear(in, out): Επίπεδο που πολλαπλασιάζει την είσοδο με πίνακα βαρών (μαθαίνει τα βάρη)
# - F.softmax(x, dim): Μετατρέπει τους βαθμούς σε πιθανότητες (με άθροισμα 1)
# - Τελεστής @: Πολλαπλασιασμός πινάκων (όπως η torch.matmul)

## 1. Γιατί δεν αρκούν οι στατικές διανυσματικές αναπαραστάσεις

Το πρόβλημα: Οι διανυσματικές αναπαραστάσεις του Κεφαλαίου 9 είναι **στατικές** — κάθε token παίρνει το ίδιο διάνυσμα ανεξάρτητα από τα συμφραζόμενα.

In [ ]:
# Προσομοίωσε την έξοδο της GPT2Embeddings από το Κεφάλαιο 9
batch_size = 2
seq_len = 6
embed_dim = 768

# Αναπαραστάσεις από το Κεφάλαιο 9 (εδώ είναι τυχαίες, αλλά φαντάσου ότι είναι οι πραγματικές)
embeddings = torch.randn(batch_size, seq_len, embed_dim)
print(f"Σχήμα των διανυσματικών αναπαραστάσεων εισόδου: {embeddings.shape}")
# Αναμενόμενη έξοδος: torch.Size([2, 6, 768])

# Αυτές είναι οι ΣΤΑΤΙΚΕΣ αναπαραστάσεις του Κεφαλαίου 9
# Στόχος μας: Να τις μετατρέψουμε σε αναπαραστάσεις που ΕΞΑΡΤΩΝΤΑΙ ΑΠΟ ΤΑ ΣΥΜΦΡΑΖΟΜΕΝΑ

print("\nΗ λέξη «bank» παίρνει πάντα το ίδιο διάνυσμα:")
print("- «river bank» → ίδια αναπαράσταση")
print("- «savings bank» → ίδια αναπαράσταση")
print("- Κι όμως, στις δύο φράσεις η λέξη σημαίνει κάτι εντελώς διαφορετικό!")

## 2. Φτιάξε τον μηχανισμό αυτοπροσοχής βήμα βήμα

Ας φτιάξουμε τον μηχανισμό προσοχής σταδιακά, βλέποντας σε κάθε βήμα το σχήμα των τανυστών.

**Η βασική ιδέα:** Κάθε token ρωτάει: «Ποια άλλα token σε βοηθούν να καταλάβεις τι σημαίνω;» 
- **Ερώτημα (query, Q)**: «Τι ψάχνω;»
- **Κλειδί (key, K)**: «Τι προσφέρω;»
- **Τιμή (value, V)**: «Η πληροφορία που έχω να μοιραστώ»

Σκέψου μια μηχανή αναζήτησης: το ερώτημα είναι αυτό που πληκτρολογείς, τα κλειδιά είναι οι τίτλοι ή οι ετικέτες των εγγράφων και οι τιμές είναι το ίδιο το περιεχόμενό τους.

### Βήμα 1: Φτιάξε τις γραμμικές προβολές για τα ερωτήματα, τα κλειδιά και τις τιμές

**Τι κάνει η `nn.Linear(in_dim, out_dim)`;**
- Το PyTorch αποθηκεύει τον πίνακα `weight` με σχήμα `(out_dim, in_dim)`.
- Η προβολή υπολογίζεται ως `input @ weight.T`: στα δεξιά της εισόδου ο πίνακας έχει σχήμα `(in_dim, out_dim)`. Αν υπάρχει όρος μετατόπισης, προστίθεται στο αποτέλεσμα.
- Το μοντέλο «μαθαίνει» αυτά τα βάρη — τα ενημερώνει κατά την εκπαίδευση

In [ ]:
# Ας χρησιμοποιήσουμε μικρότερη διάσταση για να φαίνεται πιο καθαρά το παράδειγμα
d_model = 768  # Διάσταση των αναπαραστάσεων (Κεφάλαιο 9)
d_k = 64       # Διάσταση για τα Q, K, V (συνήθως: d_model / num_heads)

# Φτιάξε τα επίπεδα προβολής (το μοντέλο μαθαίνει τις παραμέτρους τους!)
W_q = nn.Linear(d_model, d_k)  # Προβολή ερωτήματος (κρατάμε τον όρο μετατόπισης, όπως στο GPT-2)
W_k = nn.Linear(d_model, d_k)  # Προβολή κλειδιού
W_v = nn.Linear(d_model, d_k)  # Προβολή τιμής

# Εφάρμοσε τις προβολές στις αναπαραστάσεις για να πάρεις τα Q, K, V
# Γιατί χρησιμοποιούμε τη Linear; Μαθαίνει τον καλύτερο μετασχηματισμό για κάθε ρόλο
Q = W_q(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
K = W_k(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
V = W_v(embeddings)  # (batch, seq, d_k) = (2, 6, 64)

print(f"Σχήμα του Q: {Q.shape}")  # Αναμενόμενο: torch.Size([2, 6, 64])
print(f"Σχήμα του K: {K.shape}")  # Αναμενόμενο: torch.Size([2, 6, 64])
print(f"Σχήμα του V: {V.shape}")  # Αναμενόμενο: torch.Size([2, 6, 64])

print("\nΚάθε token έχει τώρα:")
print("- Διάνυσμα Q (64 διαστάσεις): «Τι ψάχνω»")
print("- Διάνυσμα K (64 διαστάσεις): «Τι προσφέρω»")
print("- Διάνυσμα V (64 διαστάσεις): «Η πληροφορία που έχω να μοιραστώ»")

### Βήμα 2: Υπολόγισε τους βαθμούς προσοχής (Q · K^T)

In [ ]:
# Υπολόγισε τους βαθμούς προσοχής: Q @ K^T
# Για να ταιριάζουν οι διαστάσεις στον πολλαπλασιασμό πινάκων, χρειάζεται να μεταθέσουμε τους δύο τελευταίους άξονες του K

# Σχήμα του Q: (batch, seq, d_k) = (2, 6, 64)
# Σχήμα του K: (batch, seq, d_k) = (2, 6, 64)

# Η K.transpose(-2, -1) ανταλλάσσει τις δύο τελευταίες διαστάσεις:
# Αρνητικοί δείκτες: -1 = τελευταία διάσταση, -2 = προτελευταία
# Έτσι αλλάζει το σχήμα του K: (2, 6, 64) → (2, 64, 6)

# Πολλαπλασιασμός πινάκων: (2, 6, 64) @ (2, 64, 6) → (2, 6, 6)
scores = Q @ K.transpose(-2, -1)

print(f"Σχήμα του Q: {Q.shape}")
print(f"Σχήμα του K: {K.shape}")
print(f"Σχήμα του K μετά τη μετάθεση: {K.transpose(-2, -1).shape}")
print(f"Σχήμα των βαθμών προσοχής: {scores.shape}")
# Αναμενόμενο: torch.Size([2, 6, 6])

print(f"\nΒαθμοί προσοχής για το πρώτο στοιχείο της παρτίδας:")
print(scores[0])
print("\nΠίνακας 6×6: το στοιχείο [i,j] είναι ο βαθμός προσοχής του token i προς το token j")

### Βήμα 3: Κλιμάκωσε τους βαθμούς προσοχής

In [ ]:
# Διαίρεσε με την τετραγωνική ρίζα της διάστασης
# Γιατί τετραγωνική ρίζα; Μαθηματικά αποδεικνύεται ότι έτσι η διακύμανση παραμένει σταθερή
scores = scores / math.sqrt(d_k)

print(f"Σχήμα των βαθμών μετά την κλιμάκωση: {scores.shape}")  # Παραμένει (2, 6, 6)
print(f"Θεωρία: αν τα στοιχεία των Q και K έχουν διακύμανση 1, η διακύμανση των βαθμών είναι περίπου {d_k}")
print(f"Μετά τη διαίρεση με sqrt({d_k}) = {math.sqrt(d_k):.2f}, η θεωρία προβλέπει διακύμανση περίπου 1")

print("\nΓιατί έχει σημασία; Χωρίς κλιμάκωση, όταν τα διανύσματα έχουν πολλές διαστάσεις,")
print("ο μηχανισμός προσοχής θα έδινε σχεδόν όλο το βάρος σε ένα μόνο token")
print("και θα έχανε το όφελος του συνδυασμού πληροφοριών από πολλά token.")

### Βήμα 4: Εφάρμοσε τη softmax για να πάρεις τα βάρη προσοχής

Αυτές οι τιμές είναι θεωρητικές· δεν τις μετρήσαμε στο παράδειγμά μας. Ο υπολογισμός προϋποθέτει ότι τα στοιχεία των Q και K έχουν διακύμανση 1. Στο παράδειγμά μας οι βαθμοί προκύπτουν από γραμμικές προβολές τυχαίων διανυσματικών αναπαραστάσεων (embeddings) με `nn.Linear`, οπότε η διακύμανση που μετράμε είναι πολύ μικρότερη από 64. Αυτό που έχει σημασία είναι η αναλογία: η κλιμάκωση διαιρεί τη διακύμανση με `d_k`, όποια κι αν είναι η αρχική της τιμή.


In [ ]:
# Εφάρμοσε τη softmax κατά μήκος της τελευταίας διάστασης (της διάστασης των κλειδιών)
# Έτσι, για κάθε γραμμή (δηλαδή για κάθε ερώτημα), το άθροισμα των βαρών είναι 1
attn_weights = F.softmax(scores, dim=-1)

print(f"Σχήμα των βαρών προσοχής: {attn_weights.shape}")  # Αναμενόμενο: (2, 6, 6)
print(f"\nΒάρη προσοχής για το token 0 (πρώτο στοιχείο της παρτίδας):")
print(attn_weights[0, 0])
# Παράδειγμα εξόδου: tensor([0.15, 0.20, 0.30, 0.18, 0.10, 0.07])
# Το άθροισμά τους είναι 1.0!

print(f"\nΆθροισμα βαρών για το token 0: {attn_weights[0, 0].sum().item():.4f}")
# Αναμενόμενο: Άθροισμα βαρών για το token 0: 1.0000

### Βήμα 5: Υπολόγισε το σταθμισμένο άθροισμα των τιμών

In [ ]:
# Βάρη προσοχής:    (batch, seq, seq) = (2, 6, 6)
# Τιμές:             (batch, seq, d_k)  = (2, 6, 64)
# Ζητούμενο σχήμα:   (batch, seq, d_k)  = (2, 6, 64)

output = attn_weights @ V

print(f"Σχήμα της εξόδου: {output.shape}")  # Αναμενόμενο: torch.Size([2, 6, 64])

print(f"\nΑρχική διανυσματική αναπαράσταση του token 0 (πρώτες 10 διαστάσεις):")
print(embeddings[0, 0, :10])

print(f"\nΈξοδος του μηχανισμού προσοχής για το token 0 (πρώτες 10 διαστάσεις):")
print(output[0, 0, :10])
print("\nΟι τιμές άλλαξαν! Η αναπαράσταση του token περιέχει πλέον πληροφορίες από τα συμφραζόμενα")

### Η πλήρης συνάρτηση προσοχής

Ας συνδυάσουμε τα 5 βήματα σε μία συνάρτηση:

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Υπολογίζει την προσοχή με κλιμακωτό εσωτερικό γινόμενο.
    
    Ορίσματα:
        Q: Ερωτήματα (batch, seq, d_k)
        K: Κλειδιά   (batch, seq, d_k)
        V: Τιμές     (batch, seq, d_k)
        mask: Προαιρετική μάσκα (batch, seq, seq)
    
    Επιστρέφει:
        output: Έξοδος του μηχανισμού προσοχής (batch, seq, d_k)
        attn_weights: Βάρη προσοχής (batch, seq, seq)
    """
    d_k = Q.size(-1)  # Βρες τη διάσταση των διανυσμάτων ερωτήματος και κλειδιού
    
    # Βήμα 1: Υπολόγισε τους βαθμούς προσοχής με το γινόμενο Q @ K^T
    scores = Q @ K.transpose(-2, -1)  # (batch, seq, seq)
    
    # Βήμα 2: Διαίρεσε τους βαθμούς με sqrt(d_k)
    scores = scores / math.sqrt(d_k)
    
    # Βήμα 3: Εφάρμοσε τη μάσκα, αν έχει δοθεί
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Βήμα 4: Εφάρμοσε τη softmax για να πάρεις τα βάρη προσοχής
    attn_weights = F.softmax(scores, dim=-1)  # (batch, seq, seq)
    
    # Βήμα 5: Υπολόγισε το σταθμισμένο άθροισμα των τιμών
    output = attn_weights @ V  # (batch, seq, d_k)
    
    return output, attn_weights


# Δοκίμασε τη συνάρτηση
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Σχήμα εξόδου: {output.shape}")  # Αναμενόμενο: (2, 6, 64)
print(f"Σχήμα των βαρών προσοχής: {attn_weights.shape}")  # Αναμενόμενο: (2, 6, 6)
print(f"\nΠώς κατανέμεται η προσοχή του πρώτου token:")
print(attn_weights[0, 0])

## 3. Αιτιακή μάσκα για αυτοπαλίνδρομη παραγωγή κειμένου

### Πώς μπορεί το μοντέλο να «κλέψει»;

Χωρίς μάσκα, όταν το μοντέλο επεξεργάζεται το «cat», μπορεί να στρέψει την προσοχή του σε ΟΛΑ τα token, ακόμη και σε όσα ακολουθούν! Έτσι «κλέβει» κατά την εκπαίδευση.

In [ ]:
def create_causal_mask(seq_len):
    """
    Δημιουργεί μια αιτιακή μάσκα: 1 στη διαγώνιο και κάτω από αυτήν, 0 πάνω από αυτήν.
    
    Επιστρέφει:
        mask: Τανυστής κινητής υποδιαστολής με σχήμα (seq_len, seq_len) και τιμές 1 (επιτρέπεται η προσοχή) και 0 (αποκλείεται η προσοχή)
    """
    # Η torch.tril δημιουργεί έναν κάτω τριγωνικό πίνακα
    # 1 κάτω από τη διαγώνιο (και στην ίδια τη διαγώνιο), 0 πάνω από τη διαγώνιο
    mask = torch.tril(torch.ones(seq_len, seq_len))
    
    return mask


# Παράδειγμα με seq_len = 6
mask = create_causal_mask(6)
print("Αιτιακή μάσκα (1 = επιτρέπεται η προσοχή, 0 = αποκλείεται η προσοχή):")
print(mask)

print("\nΠώς διαβάζεις τη μάσκα:")
print("- Γραμμή 0 (token 0): Βλέπει μόνο τη στήλη 0")
print("- Γραμμή 2 (token 2): Βλέπει τις στήλες 0, 1, 2")
print("- Γραμμή 5 (token 5): Βλέπει όλες τις στήλες από 0 έως 5")

### Εφαρμογή της μάσκας

In [ ]:
# Δοκίμασε την προσοχή με αιτιακή μάσκα
Q_test = torch.randn(2, 6, 64)  # (batch, seq, d_k)
K_test = torch.randn(2, 6, 64)
V_test = torch.randn(2, 6, 64)

# Δημιούργησε την αιτιακή μάσκα
causal_mask = create_causal_mask(6)  # (seq, seq)

# Υπολόγισε την προσοχή με τη μάσκα
output_masked, attn_weights_masked = scaled_dot_product_attention(
    Q_test, K_test, V_test, mask=causal_mask
)

print("Βάρη προσοχής ΜΕ αιτιακή μάσκα (πρώτο στοιχείο της παρτίδας):")
print(attn_weights_masked[0])

# Σε κάθε θέση που απέκλεισε η μάσκα, ο βαθμός έγινε -inf πριν από τη softmax, άρα το αντίστοιχο βάρος πρέπει να είναι μηδέν.
assert attn_weights_masked.triu(diagonal=1).count_nonzero() == 0
row_sums = attn_weights_masked.sum(dim=-1)
torch.testing.assert_close(row_sums, torch.ones_like(row_sums))
print("Η μάσκα πέρασε τον έλεγχο: κανένα token δεν βλέπει τα επόμενα και τα βάρη κάθε γραμμής έχουν άθροισμα 1.")

## 4. Προσοχή με πολλαπλές κεφαλές

### Γιατί χρειαζόμαστε πολλές κεφαλές;

Μια κεφαλή προσοχής μπορεί να εντοπίσει μόνο ΕΝΑ είδος σχέσης κάθε φορά. Με πολλές κεφαλές, το μοντέλο μπορεί να μαθαίνει διαφορετικά μοτίβα ταυτόχρονα:

| Κεφαλή | Τι μπορεί να μάθει |
|------|---------------------|
| Κεφαλή 1 | Σχέσεις υποκειμένου και ρήματος («cat» → «sat») |
| Κεφαλή 2 | Σχέσεις επιθέτου και ουσιαστικού («lazy» → «dog») |
| Κεφαλή 3 | Μοτίβα σε γειτονικές λέξεις (άμεσα συμφραζόμενα) |
| Κεφαλή 4 | Σχέσεις ανάμεσα σε απομακρυσμένες λέξεις (σε τι αναφέρεται μια αντωνυμία) |

**Βασική ιδέα:** 12 κεφαλές με 64 διαστάσεις η καθεμιά = 768 διαστάσεις συνολικά = όσες έχει 1 μεγάλη κεφαλή!
Δεν χρειάζονται επιπλέον παράμετροι — κάθε κεφαλή εξετάζει την ακολουθία από διαφορετική σκοπιά.

### Αποδοτική υλοποίηση

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Αποδοτική προσοχή με πολλαπλές κεφαλές (επεξεργάζεται όλες τις κεφαλές μαζί).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "Το d_model πρέπει να διαιρείται ακριβώς με το num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Κοινή προβολή QKV (3 φορές πιο αποδοτική από τις χωριστές προβολές!)
        # Γιατί 3 * d_model; Επειδή υπολογίζουμε ταυτόχρονα τα Q, K, V
        # Όπως και στο GPT-2, ισχύει bias=True, οπότε μπορούμε αργότερα να φορτώσουμε τους όρους μετατόπισης του προεκπαιδευμένου μοντέλου
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Προβολή εξόδου
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout στα βάρη προσοχής
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Βήμα 1: Υπολόγισε τα Q, K, V (με μία προβολή!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Βήμα 2: Χώρισε τον τανυστή σε Q, K, V και προσάρμοσε το σχήμα για τις πολλαπλές κεφαλές =====
        # Άλλαξε το σχήμα σε (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Άλλαξε τη σειρά των αξόνων σε (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Ξεχώρισε τα Q, K, V: το καθένα έχει σχήμα (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Βήμα 3: Προσοχή με κλιμακωτό εσωτερικό γινόμενο (για όλες τις κεφαλές μαζί) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Εφάρμοσε την αιτιακή μάσκα, αν έχει δοθεί
        if mask is not None:
            # Πρόσθεσε άξονες ώστε η μάσκα να εφαρμόζεται σε όλες τις κεφαλές: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Σταθμισμένο άθροισμα των τιμών
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Βήμα 4: Συνένωσε τις εξόδους των κεφαλών =====
        # Αντάλλαξε τους άξονες για να πάρεις το σχήμα (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Άλλαξε το σχήμα σε (batch, seq, d_model) για να συνενώσεις τις εξόδους των κεφαλών
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Βήμα 5: Τελική προβολή =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Κάθε κεφαλή υπολογίζει βάρη προσοχής για τις έξι θέσεις. Η συνένωση των εξόδων των δώδεκα κεφαλών επαναφέρει το πλάτος στο 768.
x_in = torch.randn(2, 6, 768)
mha = MultiHeadAttention(d_model=768, num_heads=12).eval()
with torch.no_grad():
    out, weights = mha(x_in, create_causal_mask(6))
assert out.shape == x_in.shape
assert weights.shape == (2, 12, 6, 6)
assert weights.triu(diagonal=1).count_nonzero() == 0
print("Ο έλεγχος της προσοχής με πολλαπλές κεφαλές πέτυχε:", out.shape, weights.shape)


## 5. Πλήρη μπλοκ Transformer

### Δίκτυο πρόσθιας τροφοδότησης

Το δίκτυο πρόσθιας τροφοδότησης αυξάνει το πλάτος της αναπαράστασης (768 → 3072), εφαρμόζει μια μη γραμμική συνάρτηση και έπειτα επαναφέρει το πλάτος στην αρχική του τιμή (3072 → 768). 

**Τι είναι η GELU;**
- Η GELU (Gaussian Error Linear Unit) είναι συνάρτηση ενεργοποίησης
- Μοιάζει με τη ReLU, αλλά είναι πιο ομαλή — δεν κόβει απότομα το σήμα στο μηδέν
- Χρησιμοποιείται στο GPT-2, στο BERT και στα περισσότερα σύγχρονα μοντέλα Transformer
- Σκέψου τη σαν «πύλη»: ρυθμίζει πόσο από το σήμα περνάει, ανάλογα με την τιμή της εισόδου

In [ ]:
class FeedForward(nn.Module):
    """
    Δίκτυο πρόσθιας τροφοδότησης που λειτουργεί ανά θέση.
    Επεξεργάζεται κάθε θέση ανεξάρτητα (με τα ίδια βάρη για όλες τις θέσεις).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768 για το GPT-2 small)
            d_ff: Εσωτερική διάσταση του δικτύου πρόσθιας τροφοδότησης (συνήθως 4 * d_model = 3072)
            dropout: Πιθανότητα απενεργοποίησης (dropout)
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), αύξησε την τελευταία διάσταση
        x = F.gelu(x, approximate="tanh")  # Μη γραμμική ενεργοποίηση (η παραλλαγή της GELU που χρησιμοποιεί το GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), πρόβαλε ξανά στην αρχική διάσταση
        return x


# Δοκίμασε το δίκτυο
ffn = FeedForward(d_model=768, d_ff=3072, dropout=0.1)
x_test = torch.randn(2, 6, 768)
output_ffn = ffn(x_test)

print(f"Σχήμα εισόδου:  {x_test.shape}")      # Αναμενόμενο: (2, 6, 768)
print(f"Σχήμα εξόδου:   {output_ffn.shape}")  # Αναμενόμενο: (2, 6, 768)


### Πλήρες μπλοκ Transformer (με κανονικοποίηση πριν από κάθε υποεπίπεδο)

**Κανονικοποίηση πριν ή μετά το υποεπίπεδο:**
- **Κανονικοποίηση μετά (post-norm)** (αρχικό Transformer): το LayerNorm εφαρμόζεται ΜΕΤΑ από κάθε υποεπίπεδο
- **Κανονικοποίηση πριν (pre-norm)** (GPT-2 και σύγχρονα μοντέλα): το LayerNorm εφαρμόζεται ΠΡΙΝ από κάθε υποεπίπεδο

Γιατί να κάνεις την κανονικοποίηση πριν από το υποεπίπεδο; Έτσι η εκπαίδευση είναι πιο σταθερή στα βαθιά δίκτυα (12+ επίπεδα). Οι κλίσεις διαδίδονται πιο ομαλά μέσα από τις υπολειμματικές συνδέσεις.

**Υπολειμματικές συνδέσεις:** `output = x + sublayer(x)`
- Δημιουργούν άμεσες διαδρομές για να διαδίδονται οι κλίσεις προς τα πίσω
- Χωρίς υπολειμματικές συνδέσεις, οι κλίσεις στα βαθιά δίκτυα τείνουν στο μηδέν

In [ ]:
class TransformerBlock(nn.Module):
    """
    Πλήρες μπλοκ Transformer με προσοχή με πολλαπλές κεφαλές, δίκτυο πρόσθιας τροφοδότησης,
    υπολειμματικές συνδέσεις και κανονικοποίηση επιπέδου πριν από κάθε υποεπίπεδο (όπως στο GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768)
            num_heads: Αριθμός κεφαλών προσοχής (12)
            d_ff: Εσωτερική διάσταση του δικτύου πρόσθιας τροφοδότησης (3072 = 4 * d_model)
            dropout: Πιθανότητα απενεργοποίησης
        """
        super().__init__()

        # Κανονικοποίηση επιπέδου (πριν από κάθε υποεπίπεδο)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Προσοχή με πολλαπλές κεφαλές
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Δίκτυο πρόσθιας τροφοδότησης
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (εφαρμόζεται μετά από κάθε υποεπίπεδο)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Ορίσματα:
            x: Διανυσματικές αναπαραστάσεις εισόδου (batch, seq, d_model)
            mask: Αιτιακή μάσκα (seq, seq) ή (1, 1, seq, seq)

        Επιστρέφει:
            x: Διανυσματικές αναπαραστάσεις εξόδου (batch, seq, d_model)
            attn_weights: Βάρη προσοχής (batch, heads, seq, seq)
        """
        # ===== Προσοχή με πολλαπλές κεφαλές και υπολειμματική σύνδεση =====
        # Pre-norm: κανονικοποίησε την είσοδο ΠΡΙΝ από την προσοχή
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Υπολειμματική σύνδεση

        # ===== Δίκτυο πρόσθιας τροφοδότησης με υπολειμματική σύνδεση =====
        # Pre-norm: κανονικοποίησε την είσοδο ΠΡΙΝ από το FFN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Υπολειμματική σύνδεση

        return x, attn_weights


# Το μπλοκ πρέπει να διατηρεί το σχήμα της υπολειμματικής ροής και να τηρεί τους περιορισμούς της αιτιακής μάσκας.
block = TransformerBlock(d_model=768, num_heads=12, d_ff=3072).eval()
with torch.no_grad():
    out, weights = block(x_in, create_causal_mask(6))
assert out.shape == x_in.shape
assert weights.shape == (2, 12, 6, 6)
assert weights.triu(diagonal=1).count_nonzero() == 0
print("Το μπλοκ πέρασε τον έλεγχο:", out.shape, weights.shape)


## 6. Οπτικοποίηση των μοτίβων προσοχής

In [ ]:
def visualize_attention(attn_weights, tokens, head_idx=0, ax=None):
    """
    Εμφανίζει τα βάρη προσοχής μιας συγκεκριμένης κεφαλής σε θερμικό χάρτη.
    
    Ορίσματα:
        attn_weights: Βάρη προσοχής με σχήμα (batch, heads, seq, seq)
        tokens: Λίστα με τις συμβολοσειρές των token (μήκος = seq)
        head_idx: Δείκτης της κεφαλής που θα εμφανιστεί
        ax: Άξονες της Matplotlib (αν δώσεις None, δημιουργείται νέο σχήμα)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    # Πάρε τα βάρη της επιλεγμένης κεφαλής για το πρώτο στοιχείο της παρτίδας
    weights = attn_weights[0, head_idx].detach().cpu().numpy()
    
    # Σχεδίασε τον θερμικό χάρτη
    im = ax.imshow(weights, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    
    # Όρισε τις θέσεις των ενδείξεων και τις ετικέτες τους
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    
    # Ετικέτες αξόνων και τίτλος
    ax.set_xlabel('Κλειδί (προς ποιο token στρέφεται η προσοχή)', fontsize=10)
    ax.set_ylabel('Ερώτημα (από ποιο token ξεκινά η προσοχή)', fontsize=10)
    ax.set_title(f'Βάρη προσοχής — Κεφαλή {head_idx}', fontsize=12)
    
    # Χρωματική κλίμακα
    plt.colorbar(im, ax=ax, label='Βάρος προσοχής')
    
    return ax


# Παράδειγμα: Επεξεργάσου μια πραγματική πρόταση
from transformers import AutoTokenizer

# Κωδικοποίησε την αγγλική πρόταση σε token ώστε οι ετικέτες να αντιστοιχούν στην αρχική είσοδο
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The quick brown fox jumps over the lazy dog"
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, seq)

# Βρες τη συμβολοσειρά κάθε token
tokens = [tokenizer.decode([t]) for t in token_ids[0]]
print(f"Τα token: {tokens}")

# Πέρασε τα ID από το επίπεδο διανυσματικών αναπαραστάσεων για να προσομοιώσεις την έξοδο του Κεφαλαίου 9
embed_layer = nn.Embedding(50257, 768)
embeddings_viz = embed_layer(token_ids)  # (1, seq, 768)

# Δημιούργησε την αιτιακή μάσκα
seq_len_viz = embeddings_viz.size(1)
mask_viz = create_causal_mask(seq_len_viz)

# Πέρασε τις αναπαραστάσεις από το μπλοκ Transformer
block_viz = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_viz, attn_weights_viz = block_viz(embeddings_viz, mask_viz)

print(f"\nΣχήμα του τανυστή βαρών προσοχής: {attn_weights_viz.shape}")  # Αναμενόμενο: (1, 12, seq, seq)

# Εμφάνισε τα βάρη των πρώτων 4 κεφαλών
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axes.flat):
    visualize_attention(attn_weights_viz, tokens, head_idx=i, ax=ax)

plt.tight_layout()
plt.show()

print("\nΤι βλέπεις:")
print("1. Πάνω από τη διαγώνιο όλα τα βάρη είναι μηδενικά (η αιτιακή μάσκα λειτουργεί!)")
print("2. Κάθε κεφαλή δείχνει διαφορετικό μοτίβο προσοχής (αυτή η ποικιλία είναι χρήσιμη)")
print("3. Κάποιες κεφαλές εστιάζουν σε κοντινά token, άλλες σε εξαρτήσεις μεταξύ απομακρυσμένων token")

## 7. Όλα τα βήματα της επεξεργασίας

Ας παρακολουθήσουμε τη διαδρομή των δεδομένων από το ανεπεξέργαστο κείμενο μέχρι την έξοδο του μπλοκ Transformer:

In [ ]:
# Η κλάση GPT2Embeddings από το Κεφάλαιο 9, για να μπορείς να εκτελέσεις αυτόνομα το notebook
if 'GPT2Embeddings' not in globals():
    class GPT2Embeddings(nn.Module):
        """Προσθέτει τις διανυσματικές αναπαραστάσεις (embeddings) των token και των θέσεών τους (η κλάση από το Κεφάλαιο 9)."""
        def __init__(self, vocab_size, max_seq_len, embed_dim):
            super().__init__()
            self.token_embed = nn.Embedding(vocab_size, embed_dim)
            self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
            nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
            nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
            self.max_seq_len = max_seq_len

        def forward(self, token_ids):
            batch_size, seq_len = token_ids.shape
            if seq_len > self.max_seq_len:
                raise ValueError(f"Το μήκος της ακολουθίας ({seq_len}) ξεπερνά το όριο max_seq_len ({self.max_seq_len})")
            token_embeddings = self.token_embed(token_ids)
            position_ids = torch.arange(seq_len, device=token_ids.device)
            position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
            return token_embeddings + self.pos_embed(position_ids)

# ===== Κεφάλαιο 8: Κωδικοποίηση σε token =====
from transformers import AutoTokenizer
text = "The cat sat on the mat"  # Αρχικό κείμενο: το κρατάμε στα αγγλικά για να μην αλλάξουν τα token του παραδείγματος
tokenizer = AutoTokenizer.from_pretrained("gpt2")
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, 6)

print("Βήμα 1: Κωδικοποίηση σε token")
print(f"Κείμενο: {text}")
print(f"ID των token: {token_ids}")
print(f"Σχήμα: {token_ids.shape}\n")

# ===== Κεφάλαιο 9: Διανυσματικές αναπαραστάσεις =====
# Η κλάση GPT2Embeddings από το Κεφάλαιο 9 (ορίζεται παραπάνω αν παρέλειψες εκείνο το notebook)
embed_layer = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
embeddings = embed_layer(token_ids)  # (1, 6, 768)

print("Βήμα 2: Διανυσματική αναπαράσταση")
print(f"Σχήμα του τανυστή διανυσματικών αναπαραστάσεων: {embeddings.shape}")
print(f"Διάνυσμα του πρώτου token (πρώτες 10 συνιστώσες): {embeddings[0, 0, :10]}\n")

# ===== Κεφάλαιο 10: Προσοχή (το κεφάλαιο που διαβάζεις) =====
# Δημιούργησε την αιτιακή μάσκα
seq_len_final = token_ids.size(1)  # 6
mask_final = create_causal_mask(seq_len_final)

# Μπλοκ Transformer
block_final = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_final, attn_weights_final = block_final(embeddings, mask_final)  # (1, 6, 768)

print("Βήμα 3: Προσοχή (το κεφάλαιο που διαβάζεις)")
print(f"Σχήμα της εξόδου: {output_final.shape}")
print(f"Διάνυσμα του πρώτου token μετά την προσοχή (πρώτες 10 συνιστώσες): {output_final[0, 0, :10]}\n")

print("Η επεξεργασία ολοκληρώθηκε!")
print("Αρχικό κείμενο → Token → Διανυσματικές αναπαραστάσεις → Προσοχή → Διανύσματα που λαμβάνουν υπόψη τα συμφραζόμενα")

print("\n===== Μια ματιά στο Κεφάλαιο 11: Στοίβαξε 12 μπλοκ =====")
print("Στο Κεφάλαιο 11 θα περάσουμε την έξοδο από 11 ακόμη μπλοκ Transformer!")

## Ασκήσεις

### Άσκηση 1: Υπολόγισε την προσοχή με το χέρι

Με βάση τους πίνακες Q, K, V, υπολόγισε με το χέρι τους βαθμούς προσοχής, εφάρμοσε τη softmax και χρησιμοποίησε τα βάρη που προκύπτουν για να βρεις την έξοδο. Έλεγξε ότι τα αποτελέσματά σου συμφωνούν με αυτά της `scaled_dot_product_attention()`.

In [ ]:
# Δημιούργησε απλούς πίνακες Q, K, V διαστάσεων 2×3 (batch=1, seq=2, d_k=3)
Q_ex = torch.tensor([[[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
K_ex = torch.tensor([[[1.0, 1.0, 0.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
V_ex = torch.tensor([[[2.0, 0.0, 1.0], [1.0, 2.0, 0.0]]])  # (1, 2, 3)

# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ: 
# 1. Υπολόγισε τους βαθμούς: scores = Q @ K^T
# 2. Διαίρεσε τους βαθμούς με sqrt(d_k)
# 3. Εφάρμοσε τη softmax
# 4. Υπολόγισε το σταθμισμένο άθροισμα των διανυσμάτων του V
# 5. Σύγκρινε το αποτέλεσμα με αυτό της scaled_dot_product_attention()

### Άσκηση 2: Έλεγχος της αιτιακής μάσκας

Υπολόγισε τα βάρη προσοχής με και χωρίς αιτιακή μάσκα. Βεβαιώσου ότι, όταν χρησιμοποιείς τη μάσκα, όλα τα βάρη πάνω από την κύρια διαγώνιο είναι μηδέν.

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ:
# 1. Δημιούργησε τους Q, K, V για seq_len=5
# 2. Υπολόγισε την προσοχή ΧΩΡΙΣ μάσκα
# 3. Υπολόγισε την προσοχή ΜΕ αιτιακή μάσκα
# 4. Εμφάνισε και τους δύο πίνακες βαρών προσοχής
# 5. Βεβαιώσου ότι στον πίνακα που προκύπτει με τη μάσκα όλα τα βάρη πάνω από την κύρια διαγώνιο είναι μηδέν

### Άσκηση 3: Σχήματα τανυστών στην προσοχή με πολλαπλές κεφαλές

Παρακολούθησε βήμα βήμα πώς αλλάζει το σχήμα των τανυστών στη `MultiHeadAttention`, χρησιμοποιώντας συγκεκριμένες αριθμητικές τιμές.

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ:
# Δημιούργησε έναν μηχανισμό προσοχής με πολλαπλές κεφαλές (MHA) για d_model=512, num_heads=8, seq_len=10
# Εμφάνισε το σχήμα σε κάθε βήμα:
# 1. Στην είσοδο
# 2. Μετά τις προβολές Q, K, V
# 3. Αφού αλλάξεις το σχήμα για να διαχωρίσεις τις κεφαλές
# 4. Αφού υπολογίσεις την προσοχή
# 5. Αφού συνενώσεις τις εξόδους των κεφαλών
# 6. Μετά την προβολή εξόδου

### Άσκηση 4: Σύγκριση προσοχής με μία κεφαλή και με πολλαπλές κεφαλές

Πέρασε την ίδια είσοδο από δύο μηχανισμούς προσοχής: έναν με μία κεφαλή και έναν με 12 κεφαλές. Σύγκρινε πόσες παραμέτρους έχει ο καθένας.

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ:
# 1. Δημιούργησε έναν μηχανισμό προσοχής με μία κεφαλή (d_model=768, num_heads=1)
# 2. Δημιούργησε έναν μηχανισμό προσοχής με πολλαπλές κεφαλές (d_model=768, num_heads=12)
# 3. Μέτρησε τις παραμέτρους κάθε μηχανισμού
# 4. Πέρασε την ίδια είσοδο και από τους δύο
# 5. Σύγκρινε τις εξόδους και το πλήθος των παραμέτρων

### Άσκηση 5: Οπτικοποίηση της προσοχής

Επεξεργάσου μια δική σου πρόταση και απεικόνισε τα βάρη προσοχής διαφορετικών κεφαλών.

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ:
# 1. Διάλεξε μια ενδιαφέρουσα πρόταση (π.χ. «Alice gave Bob a book»)
# 2. Χώρισέ τη σε token
# 3. Πέρασε την ακολουθία από το TransformerBlock
# 4. Απεικόνισε τα βάρη προσοχής των κεφαλών 0, 3, 7, 11
# 5. Περίγραψε τα μοτίβα που βλέπεις (τοπικά ή μεταξύ απομακρυσμένων θέσεων)

### Άσκηση 6: Μπλοκ σε σειρά

Βάλε 3 μπλοκ Transformer σε σειρά και πέρασε μια ακολουθία διαδοχικά από όλα τα μπλοκ.

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ:
# 1. Δημιούργησε 3 ξεχωριστά αντικείμενα της κλάσης TransformerBlock
# 2. Πέρασε τις διανυσματικές αναπαραστάσεις διαδοχικά από τα μπλοκ: block1 → block2 → block3
# 3. Εμφάνισε το σχήμα της εξόδου μετά από κάθε μπλοκ
# 4. Σύγκρινε τις διανυσματικές αναπαραστάσεις εισόδου με την τελική έξοδο
# 5. Πόσο διαφέρουν;

### Άσκηση 7: Κανονικοποίηση πριν ή μετά το υποεπίπεδο

Υλοποίησε ένα μπλοκ Transformer με κανονικοποίηση μετά από κάθε υποεπίπεδο (post-norm) και σύγκρινέ το με την εκδοχή που εφαρμόζει κανονικοποίηση πριν από κάθε υποεπίπεδο (pre-norm).

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ:
# 1. Υλοποίησε την κλάση TransformerBlockPostNorm έτσι ώστε το LayerNorm να εφαρμόζεται ΜΕΤΑ την υπολειμματική πρόσθεση
# 2. Πέρασε την ίδια είσοδο και από τις δύο εκδοχές (pre-norm και post-norm)
# 3. Σύγκρινε τις εξόδους
# 4. Ποια εκδοχή έχει πιο σταθερές κλίσεις; (Μπορείς να ελέγξεις τα μέτρα των κλίσεων.)

### Άσκηση 8: Πλήθος παραμέτρων

Υπολόγισε πόσες ακριβώς παραμέτρους έχει ένα μπλοκ Transformer.

In [ ]:
# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ:
# Για d_model=768, num_heads=12, d_ff=3072:
# 1. Μέτρησε τις παραμέτρους των προβολών Q, K και V
# 2. Μέτρησε τις παραμέτρους της προβολής εξόδου
# 3. Μέτρησε τις παραμέτρους του FFN (fc1 + fc2)
# 4. Μέτρησε τις παραμέτρους του LayerNorm και στα 2 επίπεδα
# 5. Πρόσθεσε τα επιμέρους πλήθη για να βρεις το σύνολο
# 6. Επαλήθευσε το αποτέλεσμα με το model.parameters()

## Σύνοψη κεφαλαίου

**Τι φτιάξαμε:**

1. Προσοχή με κλιμακωτό εσωτερικό γινόμενο (Q, K, V → βαθμοί προσοχής → softmax → σταθμισμένο άθροισμα)
2. Αιτιακή μάσκα για αυτοπαλίνδρομη παραγωγή (κάθε θέση βλέπει μόνο το δικό της token και τα προηγούμενα)
3. Αποδοτική προσοχή με πολλαπλές κεφαλές (1 κεφαλή → 12 κεφαλές αλλάζοντας το σχήμα των τανυστών)
4. Πλήρη μπλοκ Transformer (προσοχή + FFN + υπολειμματικές συνδέσεις + κανονικοποίηση επιπέδου)
5. Οπτικοποίηση της προσοχής (θερμικοί χάρτες που δείχνουν σε ποια token δίνει προσοχή το μοντέλο)

**Βασικές έννοιες:**

- **Στατικές διανυσματικές αναπαραστάσεις** (Κεφάλαιο 9) → **Αναπαραστάσεις που εξαρτώνται από τα συμφραζόμενα** (Κεφάλαιο 10)
- **Ερώτημα/κλειδί/τιμή**: Τρεις γραμμικές προβολές με διαφορετικούς ρόλους, τις οποίες μαθαίνει το μοντέλο
- **Βάρη προσοχής**: Πιθανότητες που δίνει η softmax και που δείχνουν πόσο σχετικό είναι κάθε κλειδί με το ερώτημα
- **Προσοχή με πολλαπλές κεφαλές**: Διαφορετικές κεφαλές μαθαίνουν διαφορετικά μοτίβα (χωρίς επιπλέον παραμέτρους!)
- **Υπολειμματικές συνδέσεις και κανονικοποίηση**: Επιτρέπουν την εκπαίδευση βαθιών δικτύων (100+ επίπεδα)

**Στη συνέχεια:** Στο Κεφάλαιο 11 θα βάλεις αυτά τα μπλοκ σε σειρά και θα προσθέσεις την κεφαλή πρόβλεψης του επόμενου token για να φτιάξεις ένα πλήρες μοντέλο GPT!